# MRSI Transform and Write (FFT1000 Only)

This notebook is the single producer for all processed datasets used in cPCA.

Outputs written by branch:
- baseline
- water_scaled
- water_regressed

Input rule:
- load only files ending in `complex_FFT_1000.nii`

In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import nibabel as nib
import numpy as np

In [ ]:
@dataclass(frozen=True)
class SubjectConfig:
    subject_id: str
    subject_index: int


DATA_ROOT = Path('/data/MRSI/Naren2/FURI2/mayomrs_aligned')
DATA_ROOT_CANDIDATES = [
    DATA_ROOT,
    Path('/data/MRSI/Naren2/FURI2/mayomrs_aligned'),
    Path('/data/MRSI/Naren2/FURI2'),
    Path('/data/MRSI/Naren2'),
]
AUTO_RESOLVE_DATA_ROOT = True

OUTPUT_ROOT = Path('/home/ben/MRSI_analysis/artifacts/processed_fft1000')
SUBJECTS = [
    SubjectConfig('RMS01', 1),
    SubjectConfig('RMS02', 2),
    SubjectConfig('RMS03', 3),
    SubjectConfig('RMS05', 4),
]

BACKGROUND_SUM_THRESHOLD = 1e-3
EPSILON = 1e-8
WATER_HALF_WIDTH_BINS = 12

In [ ]:
def assert_fft1000_path(path: Path) -> None:
    if not path.name.endswith('complex_FFT_1000.nii'):
        raise ValueError(f'Input is not FFT1000 constrained: {path}')


def find_single_fft1000_file(group: str, subject_id: str) -> Path:
    subject_dir = DATA_ROOT / group / subject_id
    candidates = sorted(subject_dir.glob('*complex_FFT_1000.nii'))
    if len(candidates) != 1:
        raise FileNotFoundError(
            f'Expected exactly one *complex_FFT_1000.nii in {subject_dir}, found {len(candidates)}'
        )
    path = candidates[0]
    assert_fft1000_path(path)
    return path


def find_label_file(subject_id: str) -> Path:
    # Labels are used only for WMH/BACK masks and subject label assignment.
    subject_dir = DATA_ROOT / 'REF' / subject_id
    patterns = [
        '*arr_hy_wlabels_reshaped_labelswspectra*.nii',
        '*labelswspectra*.nii'
    ]
    candidates: List[Path] = []
    for pattern in patterns:
        candidates.extend(subject_dir.glob(pattern))
    candidates = sorted(set(candidates))
    if len(candidates) == 0:
        raise FileNotFoundError(f'No label file found in {subject_dir}')
    return candidates[0]


def load_real_fft1000(path: Path) -> np.ndarray:
    data = np.array(nib.load(str(path)).dataobj)
    reshaped = data.reshape(131072, 1000, 2)
    return reshaped[:, :, 0].astype(np.float64)


def load_label_matrix(path: Path) -> np.ndarray:
    data = np.array(nib.load(str(path)).dataobj)
    return data.reshape(131072, 2049)


def split_wmh_back(label_matrix: np.ndarray, spectra: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    wmh_mask = label_matrix[:, -1] != 0
    back_mask = ~wmh_mask
    labels = label_matrix[wmh_mask][:, -1]
    return spectra[wmh_mask], spectra[back_mask], wmh_mask, labels


def derive_water_window_indices(reference_stack: np.ndarray, half_width: int = WATER_HALF_WIDTH_BINS) -> np.ndarray:
    mean_ref = reference_stack.mean(axis=0)
    peak_idx = int(np.argmax(mean_ref))
    lo = max(0, peak_idx - half_width)
    hi = min(reference_stack.shape[1], peak_idx + half_width + 1)
    return np.arange(lo, hi, dtype=int)


def robust_water_amplitude(reference_spectra: np.ndarray, water_idx: np.ndarray) -> np.ndarray:
    window = reference_spectra[:, water_idx]
    # Use a high percentile for robustness against local spikes.
    return np.percentile(window, 95, axis=1)


def scale_by_water(metabolite_spectra: np.ndarray, reference_spectra: np.ndarray, water_idx: np.ndarray, eps: float = EPSILON) -> np.ndarray:
    amplitude = robust_water_amplitude(reference_spectra, water_idx)
    safe_amp = np.maximum(amplitude, eps)
    return metabolite_spectra / safe_amp[:, None]


def build_reference_basis(reference_spectra: np.ndarray, water_idx: np.ndarray, n_components: int = 1) -> np.ndarray:
    window = reference_spectra[:, water_idx]
    centered = window - window.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    return vt[:n_components, :]


def regress_and_zero_water(metabolite_spectra: np.ndarray, basis_window: np.ndarray, water_idx: np.ndarray) -> np.ndarray:
    out = metabolite_spectra.copy()
    window = out[:, water_idx]
    B = basis_window.T
    coeffs, _, _, _ = np.linalg.lstsq(B, window.T, rcond=None)
    fitted_window = (B @ coeffs).T
    out[:, water_idx] = window - fitted_window
    # Requirement: force whole water region to zero.
    out[:, water_idx] = 0.0
    return out


def filter_background(back: np.ndarray, threshold: float = BACKGROUND_SUM_THRESHOLD) -> Tuple[np.ndarray, np.ndarray]:
    sums = back.sum(axis=1)
    mask = sums >= threshold
    return back[mask], mask


def write_branch(branch_dir: Path, wmh: np.ndarray, back: np.ndarray, labels: np.ndarray, metadata: Dict) -> None:
    branch_dir.mkdir(parents=True, exist_ok=True)
    np.save(branch_dir / 'WMH.npy', wmh)
    np.save(branch_dir / 'BACK.npy', back)
    np.save(branch_dir / 'Labels.npy', labels)
    with open(branch_dir / 'metadata.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)


def artifact_metadata_template(branch_name: str, water_idx: np.ndarray) -> Dict:
    return {
        'branch_name': branch_name,
        'input_constraint': 'complex_FFT_1000.nii only',
        'water_window_indices': water_idx.tolist(),
        'background_sum_threshold': BACKGROUND_SUM_THRESHOLD
    }

In [ ]:
per_subject = []

for subject in SUBJECTS:
    met_path = find_single_fft1000_file('METABOLITE', subject.subject_id)
    ref_path = find_single_fft1000_file('REF', subject.subject_id)
    label_path = find_label_file(subject.subject_id)

    met = load_real_fft1000(met_path)
    ref = load_real_fft1000(ref_path)
    label_matrix = load_label_matrix(label_path)

    wmh_met, back_met, wmh_mask, _ = split_wmh_back(label_matrix, met)
    wmh_ref = ref[wmh_mask]
    back_ref = ref[~wmh_mask]

    labels = np.full(wmh_met.shape[0], subject.subject_index, dtype=np.int32)

    per_subject.append({
        'subject_id': subject.subject_id,
        'met_path': str(met_path),
        'ref_path': str(ref_path),
        'label_path': str(label_path),
        'wmh_met': wmh_met,
        'back_met': back_met,
        'wmh_ref': wmh_ref,
        'back_ref': back_ref,
        'labels': labels
    })

print(f'Subjects loaded: {len(per_subject)}')
for row in per_subject:
    print(
        row['subject_id'],
        row['wmh_met'].shape,
        row['back_met'].shape,
        Path(row['met_path']).name
    )

In [ ]:
all_ref_for_window = np.concatenate([
    np.concatenate([row['wmh_ref'], row['back_ref']], axis=0)
    for row in per_subject
], axis=0)
water_idx = derive_water_window_indices(all_ref_for_window)
print(f'Water window size: {water_idx.size}')
print(f'Water window index range: {water_idx.min()} to {water_idx.max()}')

In [ ]:
wmh_baseline = np.concatenate([row['wmh_met'] for row in per_subject], axis=0)
back_baseline = np.concatenate([row['back_met'] for row in per_subject], axis=0)
labels_baseline = np.concatenate([row['labels'] for row in per_subject], axis=0)

wmh_scaled_parts = []
back_scaled_parts = []
wmh_regressed_parts = []
back_regressed_parts = []

for row in per_subject:
    wmh_scaled_parts.append(scale_by_water(row['wmh_met'], row['wmh_ref'], water_idx))
    back_scaled_parts.append(scale_by_water(row['back_met'], row['back_ref'], water_idx))

    ref_all = np.concatenate([row['wmh_ref'], row['back_ref']], axis=0)
    basis_window = build_reference_basis(ref_all, water_idx, n_components=1)
    wmh_regressed_parts.append(regress_and_zero_water(row['wmh_met'], basis_window, water_idx))
    back_regressed_parts.append(regress_and_zero_water(row['back_met'], basis_window, water_idx))

wmh_scaled = np.concatenate(wmh_scaled_parts, axis=0)
back_scaled = np.concatenate(back_scaled_parts, axis=0)
wmh_regressed = np.concatenate(wmh_regressed_parts, axis=0)
back_regressed = np.concatenate(back_regressed_parts, axis=0)

back_baseline, mask_base = filter_background(back_baseline)
back_scaled, mask_scaled = filter_background(back_scaled)
back_regressed, mask_reg = filter_background(back_regressed)

print('Baseline WMH/BACK:', wmh_baseline.shape, back_baseline.shape)
print('Scaled WMH/BACK:', wmh_scaled.shape, back_scaled.shape)
print('Regressed WMH/BACK:', wmh_regressed.shape, back_regressed.shape)
print('Labels:', labels_baseline.shape)

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

write_branch(
    OUTPUT_ROOT / 'baseline',
    wmh_baseline,
    back_baseline,
    labels_baseline,
    artifact_metadata_template('baseline', water_idx)
)

write_branch(
    OUTPUT_ROOT / 'water_scaled',
    wmh_scaled,
    back_scaled,
    labels_baseline,
    artifact_metadata_template('water_scaled', water_idx)
)

write_branch(
    OUTPUT_ROOT / 'water_regressed',
    wmh_regressed,
    back_regressed,
    labels_baseline,
    artifact_metadata_template('water_regressed', water_idx)
)

manifest = {
    'root': str(OUTPUT_ROOT),
    'branches': ['baseline', 'water_scaled', 'water_regressed'],
    'subjects': [row['subject_id'] for row in per_subject],
    'constraint': 'complex_FFT_1000.nii only',
    'water_window_indices': water_idx.tolist()
}

with open(OUTPUT_ROOT / 'manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print(f'Artifacts written to: {OUTPUT_ROOT}')